
# Practical 5: Hyperparameter Tuning with Weights & Biases (W&B)


### Why this practical?
In Practicals 1–4 you built and improved image classifiers on the **Flowers** dataset.  
Here we keep the **same dataset and pipeline** and learn how to use **W&B** to:
- Track training runs (loss/accuracy over time)
- Compare runs easily
- Try small **hyperparameter sweeps** to find better settings

**W&B website:** https://wandb.ai/  
**Beginner playlist:** https://www.youtube.com/playlist?list=PLD80i8An1OEGajeVo15ohAQYF1Ttle0lk


## Learning goals
- Log metrics to **W&B**
- Define a **small sweep** (2–6 runs)
- Pick the best hyperparameters and **retrain once**
- Keep training short in class (**10 epochs**). *If you increase epochs later, results may improve.*



## 0) Setup (one time)
1) Create a free account at W&B and copy your API key: https://wandb.ai/authorize  
2) Install packages if needed.


In [ ]:

# If running in a fresh environment, uncomment:
# !pip install -q wandb tensorflow matplotlib

import os, json
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Try to import W&B
try:
    import wandb
    from wandb.keras import WandbCallback
except Exception as e:
    print("wandb is not installed yet. Install with: pip install wandb")
    wandb, WandbCallback = None, None

WANDB_PROJECT = "cv-roadmap-practical5"
WANDB_ENTITY = None  # (optional) your username or team


In [ ]:

def wandb_login_or_disable():
    """Try to login to W&B; if it fails, run in disabled mode so the notebook still works."""
    if wandb is None:
        os.environ['WANDB_MODE'] = 'disabled'
        print("W&B not installed; running with logging disabled.")
        return False
    try:
        wandb.login()  # uses existing API key if set; otherwise opens a prompt
        os.environ.pop("WANDB_MODE", None)  # ensure online
        print("W&B online logging enabled.")
        return True
    except Exception:
        os.environ['WANDB_MODE'] = 'disabled'
        print("W&B login skipped or failed; running with logging disabled.")
        return False

_ = wandb_login_or_disable()



## 1) Use the SAME dataset and loader as Practicals 2–4 (Flowers)

We read image paths and labels from **Google Cloud CSV files** and decode images.  
Class names and image size are unchanged.


In [ ]:

# Image size and classes (same as before)
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_CHANNELS = 3
CLASS_NAMES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

# Utilities copied from earlier practicals
def read_and_decode(filename, resize_dims):
    img_bytes = tf.io.read_file(filename)
    img = tf.image.decode_jpeg(img_bytes, channels=IMG_CHANNELS)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, resize_dims)
    return img

def parse_csvline(csv_line):
    record_default = ["", ""]
    filename, label_string = tf.io.decode_csv(csv_line, record_default)
    img = read_and_decode(filename, [IMG_HEIGHT, IMG_WIDTH])
    label = tf.argmax(tf.math.equal(CLASS_NAMES, label_string))
    return img, label

# Build tf.data pipelines (same source CSVs as earlier)
BATCH_SIZE_DEFAULT = 16

train_dataset = (
    tf.data.TextLineDataset("gs://cloud-ml-data/img/flower_photos/train_set.csv")
    .map(parse_csvline, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE_DEFAULT)
    .prefetch(tf.data.AUTOTUNE)
)

eval_dataset = (
    tf.data.TextLineDataset("gs://cloud-ml-data/img/flower_photos/eval_set.csv")
    .map(parse_csvline, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE_DEFAULT)
    .prefetch(tf.data.AUTOTUNE)
)



## 2) A small, simple CNN

We keep the model **tiny** so runs finish quickly.  
Hyperparameters we will tune: **learning_rate, optimizer, dropout, batch_size**.


In [ ]:

def build_simple_cnn(dropout=0.3):
    inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
    return keras.Model(inputs, outputs, name="simple_cnn")



## 3) Training function with W&B logging

- Logs: loss/accuracy for train and validation  
- Uses **EarlyStopping** to keep time short  
- Adds `WandbCallback` when online


In [ ]:

def train_once(config):
    # Unpack config
    lr = config.get('learning_rate', 1e-3)
    opt_name = config.get('optimizer', 'adam')
    dropout = config.get('dropout', 0.3)
    batch_size = config.get('batch_size', BATCH_SIZE_DEFAULT)
    epochs = config.get('epochs', 10)  # We run only 10 epochs in class

    # Prepare datasets (rebatch if needed)
    train_ds = train_dataset.unbatch().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    val_ds = eval_dataset.unbatch().batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # Build and compile
    model = build_simple_cnn(dropout=dropout)
    if opt_name.lower() == 'adam':
        opt = keras.optimizers.Adam(lr)
    elif opt_name.lower() == 'sgd':
        opt = keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=True)
    else:
        opt = keras.optimizers.Adam(lr)

    model.compile(optimizer=opt,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # W&B run
    run = None
    if wandb is not None and os.environ.get("WANDB_MODE") != "disabled":
        run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, config=config, reinit=True)
        callbacks = [WandbCallback(save_model=False)]
    else:
        callbacks = []

    callbacks.append(keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy'))

    history = model.fit(train_ds,
                        validation_data=val_ds,
                        epochs=epochs,
                        verbose=2,
                        callbacks=callbacks)

    # Evaluate
    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    if run is not None:
        wandb.log({'final_val_loss': float(val_loss), 'final_val_accuracy': float(val_acc)})
        run.finish()

    return model, history.history, {'val_loss': float(val_loss), 'val_accuracy': float(val_acc)}



## 4) Quick single run
We train **one** model to confirm everything works.  
**Note:** We run **10 epochs** for speed. If you increase epochs, results **may improve**.


In [ ]:

default_config = {
    'learning_rate': 1e-3,
    'optimizer': 'adam',
    'dropout': 0.3,
    'batch_size': 16,
    'epochs': 10  # short for class
}

model, hist, metrics = train_once(default_config)
print(metrics)



### Plot training curves


In [ ]:

plt.figure(figsize=(6,4))
plt.plot(hist['loss'], label='train_loss')
plt.plot(hist['val_loss'], label='val_loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('Loss (10 epochs)'); plt.show()

plt.figure(figsize=(6,4))
plt.plot(hist['accuracy'], label='train_acc')
plt.plot(hist['val_accuracy'], label='val_acc')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.title('Accuracy (10 epochs)'); plt.show()

print("Reminder: we trained only 10 epochs here; more epochs may improve results.")



## 5) A **small** W&B Sweep (simple and fast)
We search only a few options to keep things **easy** and **quick**:
- `learning_rate`: [1e-3, 5e-4]
- `optimizer`: ['adam', 'sgd']
- `dropout`: [0.3, 0.4]
- `batch_size`: [16, 32]
- `epochs`: fixed at 10 (short demo)

We will run **4–6 trials** total.


In [ ]:

sweep_config = {
    "name": "flowers-simple-sweep",
    "method": "random",
    "metric": {"name": "val_accuracy", "goal": "maximize"},
    "parameters": {
        "learning_rate": {"values": [1e-3, 5e-4]},
        "optimizer": {"values": ["adam", "sgd"]},
        "dropout": {"values": [0.3, 0.4]},
        "batch_size": {"values": [16, 32]},
        "epochs": {"values": [10]}
    }
}

print(json.dumps(sweep_config, indent=2))



### Launch the sweep
- **Online mode:** create sweep + run an agent for a few trials.  
- **Disabled mode:** we manually try 4 small configs.


In [ ]:

def sweep_objective():
    cfg = dict(wandb.config) if (wandb is not None and wandb.run is not None) else dict(default_config)
    _m, _h, _metrics = train_once(cfg)

max_trials = 4  # keep it tiny for class time

if wandb is not None and os.environ.get("WANDB_MODE") != "disabled":
    sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT, entity=WANDB_ENTITY)
    wandb.agent(sweep_id, function=sweep_objective, count=max_trials)
else:
    print("W&B disabled — running 4 manual trials instead...")
    manual_space = [
        {'learning_rate': 1e-3, 'optimizer': 'adam', 'dropout': 0.3, 'batch_size': 16, 'epochs': 10},
        {'learning_rate': 5e-4, 'optimizer': 'adam', 'dropout': 0.3, 'batch_size': 32, 'epochs': 10},
        {'learning_rate': 1e-3, 'optimizer': 'sgd',  'dropout': 0.4, 'batch_size': 16, 'epochs': 10},
        {'learning_rate': 5e-4, 'optimizer': 'sgd',  'dropout': 0.4, 'batch_size': 32, 'epochs': 10},
    ]
    results_manual = []
    for i, cfg in enumerate(manual_space[:max_trials], 1):
        print(f"\n[Manual Trial {i}] {cfg}")
        _m, _h, _metrics = train_once(cfg)
        results_manual.append((_metrics['val_accuracy'], cfg))
    # Show simple summary
    results_manual.sort(key=lambda x: x[0], reverse=True)
    print("\nManual trials ranked by val_accuracy:")
    for acc, cfg in results_manual:
        print(f"acc={acc:.4f} -> {cfg}")



## 6) Retrain once with your best settings
- On the W&B website, sort runs by **val_accuracy** and copy the winning config below.  
- If offline, choose the best settings from the manual trials printed above.


In [ ]:

best_config = dict(default_config)
# Example (replace these with your best after checking W&B or manual summary):
best_config.update({'learning_rate': 5e-4, 'optimizer': 'adam', 'dropout': 0.3, 'batch_size': 32, 'epochs': 10})

print("Using:", best_config)
best_model, best_hist, best_metrics = train_once(best_config)
print(best_metrics)



## 7) Other Options Beyond W&B
While **Weights & Biases (W&B)** is a very popular tool for experiment tracking and hyperparameter tuning,  
there are other options you can explore in the future, such as:
- **TensorBoard** (comes with TensorFlow)
- **Optuna** (powerful hyperparameter optimization library)
- **KerasTuner** (easy to integrate with Keras/TensorFlow models)
- **MLflow** (experiment tracking and model registry)

In this practical, we focus only on W&B because it offers a smooth, visual, and beginner‑friendly workflow.



## 8) What's Next?
In the **next practical**, we will learn about:
- **Evaluation metrics** for different Computer Vision tasks:
  - **Classification** (accuracy, precision, recall, F1-score, confusion matrix, etc.)
  - **Object Detection** (mAP, IoU, etc.)
  - **Image Segmentation** (IoU, Dice coefficient, etc.)

We will see how to calculate these metrics in Python and interpret them for better decision‑making.



---
> *This Practical was designed to give you hands-on experience and a clear understanding of Hyperparameter Tuning concepts, including Experiment Tracking, Parameter Sweeps, and Model Selection using W&B.*
